# **💁🏻🗨️💁🏻‍♂️대화 요약 Baseline code**
> **Dialogue Summarization** 경진대회에 오신 여러분 환영합니다! 🎉    
> 본 대회에서는 최소 2명에서 최대 7명이 등장하여 나누는 대화를 요약하는 BART 기반 모델의 baseline code를 제공합니다.     
> 주어진 데이터를 활용하여 일상 대화에 대한 요약을 효과적으로 생성하는 모델을 만들어봅시다!

> **수정 사항**:
> 1. `compute_metrics` skip_special_tokens=True 적용
> 2. `inference()` 파일명 저장 버그 수정
> 3. 실패 케이스 분석 함수 추가
> 4. Optuna best 파라미터 자동 반영 코드 추가

| 섹션 | 내용 |
|------|------|
| 0 | 환경설정 & Config |
| 1 | 데이터 가공 & Dataset |
| 2 | Trainer & Training |
| 3 | Optuna 하이퍼파라미터 탐색 |
| 4 | 모델 학습 (main) |
| 5 | Test 추론 |
| 6 | Dev 추론 & 로컬 ROUGE 평가 |
| 7 | 실패 케이스 분석 

## ⚙️ 데이터 및 환경설정

### 1) 필요한 라이브러리 설치

- 필요한 라이브러리를 설치한 후 불러옵니다.

In [1]:
import pandas as pd
from datetime import datetime
import os
import re
import json
import yaml
from glob import glob
from tqdm import tqdm
from pprint import pprint
import torch
import pytorch_lightning as pl
from rouge import Rouge # 모델의 성능을 평가하기 위한 라이브러리입니다.

from torch.utils.data import Dataset , DataLoader
from transformers import AutoTokenizer, BartForConditionalGeneration, BartConfig
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers import Trainer, TrainingArguments
from transformers import EarlyStoppingCallback

import wandb # 모델 학습 과정을 손쉽게 Tracking하고, 시각화할 수 있는 라이브러리입니다.

In [2]:
# wandb API 키 설정
# 86자리 키 입력
os.environ["WANDB_API_KEY"] = "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"
os.environ["WANDB_MODE"] = "online"
os.environ["CUDA_LAUNCH_BLOCKING"]="1" # CUDA 디버깅

In [3]:
print(f'PyTorch: {torch.__version__}')
print(f'CUDA 사용 가능: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

PyTorch: 2.1.0
CUDA 사용 가능: True
GPU: NVIDIA GeForce RTX 3090


### 2) Config file 만들기 (선택)
> ⭐️ 매 실험마다 `exp_number`와 `wandb name`만 변경하면 됩니다.
- 모델 생성에 필요한 다양한 매개변수 정보를 저장할 수 있습니다.  
  따라서, 코드 상에서 모델의 매개변수를 설정할 수도 있지만 독립적인 매개변수 정보 파일을 생성하여 관리할 수 있습니다.

In [4]:
# config 설정에 tokenizer 모듈이 사용되므로 미리 tokenizer를 정의해줍니다.
tokenizer = AutoTokenizer.from_pretrained("digit82/kobart-summarization")

In [5]:
config_data = {
    "exp_number": "00", # ⭐️ 실험번호 (매 실험마다 변경 필요)

    "general": {
        "data_path": "../data/", # 모델 생성에 필요한 데이터 경로를 사용자 환경에 맞게 지정합니다.
        "model_name": "digit82/kobart-summarization", # 불러올 모델의 이름을 사용자 환경에 맞게 지정할 수 있습니다.
        "output_dir": "./" # 모델의 최종 출력 값을 저장할 경로를 설정합니다.
    },
    "tokenizer": {
        "encoder_max_len": 256, # ✅512->256 (95th pct=295토큰, 메모리절약)
        "decoder_max_len": 50, # ✅100->50 (99% 커버리지=40단어, 여유분 포함)
        "bos_token": f"{tokenizer.bos_token}",
        "eos_token": f"{tokenizer.eos_token}",
        # 특정 단어들이 분해되어 tokenization이 수행되지 않도록 special_tokens을 지정해줍니다.
        "special_tokens": [
            # 기존 6개
            '#Person1#', '#Person2#', '#Person3#', 
            '#PhoneNumber#', '#Address#', '#PassportNumber#',
            
            # ✅ EDA로 발견한 추가 토큰 (16개)
            '#Person4#', '#Person5#', '#Person6#', '#Person7#',
            # 개인정보 마스킹 토큰 (8개)
            '#DateOfBirth#', '#CardNumber#', '#Email#',
            '#CarNumber#', '#SSN#', '#Name#', '#PersonName#',
            # 새로운 개인정보 마스킹 토큰 발견
            '#Price#',

            # 고유명사 토큰 (등장 횟수 적지만 추가 권장)
            # 원래는 #Name#으로 개인정보 마스킹 되어야했던 부분
            '#Bob#', '#Kristin#', '#Alex#', '#Liliana#'
            ]
    },
    'training': {
        'overwrite_output_dir'        : True,
        'num_train_epochs'            : 20,
        'learning_rate'               : 1e-5,
        'per_device_train_batch_size' : 32,
        'per_device_eval_batch_size'  : 32,
        'warmup_ratio'                : 0.1,
        'weight_decay'                : 0.01,
        'lr_scheduler_type'           : 'cosine',
        'optim'                       : 'adamw_torch',
        'gradient_accumulation_steps' : 1,
        'evaluation_strategy'         : 'epoch',
        'save_strategy'               : 'epoch',
        'save_total_limit'            : 5,
        'fp16'                        : True,
        'load_best_model_at_end'      : True,
        'seed'                        : 42,
        'logging_dir'                 : './logs',
        'logging_strategy'            : 'epoch',
        'predict_with_generate'       : True,
        'generation_max_length'       : 50,   # ✅ 100->50
        'do_train'                    : True,
        'do_eval'                     : True,
        'early_stopping_patience'     : 3,
        'early_stopping_threshold'    : 0.001,
        'report_to'                   : 'wandb', # (선택) ✅ none ->  wandb를 사용할 때 설정합니다.
        'metric_for_best_model'       : 'rouge_avg',  # ✅ eval_loss -> rouge_avg
        'greater_is_better'           : True,
    },
    # (선택) wandb 홈페이지에 가입하여 얻은 정보를 기반으로 작성합니다.
    "wandb": {
        "entity": "s202010741-",            # ✅ 팀 wandb entity
        "project": "nlp-dialogue-summary",
        "name": "EXP00-baseline"            # ⭐️ 실험마다 변경
    },
    "inference": {
        "ckt_path": "model ckt path", # 사전 학습이 진행된 모델의 checkpoint를 저장할 경로를 설정합니다.
        "result_path": "./prediction/",
        "no_repeat_ngram_size": 3,    # 생성 설정 ✅2->3 (Abstractive 성향 대응)
        "early_stopping": True,
        "generate_max_length": 50, # ✅ 100->50
        "num_beams": 4,
        "batch_size" : 32,
        # 정확한 모델 평가를 위해 제거할 불필요한 생성 토큰들을 정의합니다.
        "remove_tokens": ['<usr>', f"{tokenizer.bos_token}", f"{tokenizer.eos_token}", f"{tokenizer.pad_token}"]
    }
}

- 참고✅    
: wandb 라이브러리를 사용하기 위해선 entity, project, name를 지정해주어야 합니다. wandb 홈페이지에 가입한 후 얻은 정보를 입력하여 작동할 수 있습니다.

In [6]:
# 모델의 구성 정보를 YAML 파일로 저장합니다.
config_path = "./config.yaml"
with open(config_path, 'w') as f:
    yaml.dump(config_data, f, allow_unicode=True, default_flow_style=False)
print('config 저장 완료')

config 저장 완료


### 3) Configuration 불러오기

In [7]:
# 저장된 config 파일을 불러옵니다.
config_path = "./config.yaml"

with open(config_path, "r") as f:
    loaded_config = yaml.load(f, Loader=yaml.FullLoader)

# wandb entity 확인
loaded_config['wandb']['entity'] = 's202010741-'

print('special_tokens 개수:', len(loaded_config['tokenizer']['special_tokens']))
print('special_tokens:', loaded_config['tokenizer']['special_tokens'])

# 불러온 config 파일의 전체 내용을 확인합니다.
pprint(loaded_config)

special_tokens 개수: 22
special_tokens: ['#Person1#', '#Person2#', '#Person3#', '#PhoneNumber#', '#Address#', '#PassportNumber#', '#Person4#', '#Person5#', '#Person6#', '#Person7#', '#DateOfBirth#', '#CardNumber#', '#Email#', '#CarNumber#', '#SSN#', '#Name#', '#PersonName#', '#Price#', '#Bob#', '#Kristin#', '#Alex#', '#Liliana#']
{'exp_number': '00',
 'general': {'data_path': '../data/',
             'model_name': 'digit82/kobart-summarization',
             'output_dir': './'},
 'inference': {'batch_size': 32,
               'ckt_path': 'model ckt path',
               'early_stopping': True,
               'generate_max_length': 50,
               'no_repeat_ngram_size': 3,
               'num_beams': 4,
               'remove_tokens': ['<usr>', '<s>', '</s>', '<pad>'],
               'result_path': './prediction/'},
 'tokenizer': {'bos_token': '<s>',
               'decoder_max_len': 50,
               'encoder_max_len': 256,
               'eos_token': '</s>',
               'special

In [8]:
# 실험에 쓰일 데이터의 경로, 사용될 모델, 모델의 최종 출력 결과를 저장할 경로에 대해 확인합니다.
# 데이터 경로 확인 (필요 시 수정)
loaded_config['general']

{'data_path': '../data/',
 'model_name': 'digit82/kobart-summarization',
 'output_dir': './'}

In [9]:
# 이곳에 사용자가 저장한 데이터 dir 설정하기
# loaded_config['general']['data_path'] = "data_path"

In [10]:
# 데이터 전처리를 하기 위해 tokenization 과정에서 필요한 정보들을 확인합니다.
loaded_config['tokenizer']

{'bos_token': '<s>',
 'decoder_max_len': 50,
 'encoder_max_len': 256,
 'eos_token': '</s>',
 'special_tokens': ['#Person1#',
  '#Person2#',
  '#Person3#',
  '#PhoneNumber#',
  '#Address#',
  '#PassportNumber#',
  '#Person4#',
  '#Person5#',
  '#Person6#',
  '#Person7#',
  '#DateOfBirth#',
  '#CardNumber#',
  '#Email#',
  '#CarNumber#',
  '#SSN#',
  '#Name#',
  '#PersonName#',
  '#Price#',
  '#Bob#',
  '#Kristin#',
  '#Alex#',
  '#Liliana#']}

In [11]:
# 모델이 훈련 시 적용될 매개변수를 확인합니다.
loaded_config['training']

{'do_eval': True,
 'do_train': True,
 'early_stopping_patience': 3,
 'early_stopping_threshold': 0.001,
 'evaluation_strategy': 'epoch',
 'fp16': True,
 'generation_max_length': 50,
 'gradient_accumulation_steps': 1,
 'greater_is_better': True,
 'learning_rate': 1e-05,
 'load_best_model_at_end': True,
 'logging_dir': './logs',
 'logging_strategy': 'epoch',
 'lr_scheduler_type': 'cosine',
 'metric_for_best_model': 'rouge_avg',
 'num_train_epochs': 20,
 'optim': 'adamw_torch',
 'overwrite_output_dir': True,
 'per_device_eval_batch_size': 32,
 'per_device_train_batch_size': 32,
 'predict_with_generate': True,
 'report_to': 'wandb',
 'save_strategy': 'epoch',
 'save_total_limit': 5,
 'seed': 42,
 'warmup_ratio': 0.1,
 'weight_decay': 0.01}

In [12]:
# 모델 학습 과정에 대한 정보를 제공해주는 wandb 설정 내용을 확인합니다.
loaded_config['wandb']

{'entity': 's202010741-',
 'name': 'EXP00-baseline',
 'project': 'nlp-dialogue-summary'}

In [13]:
# (선택) 이곳에 사용자가 사용할 wandb config 설정
loaded_config['wandb']['entity'] = "s202010741-"           # 사용할 wandb repo name
loaded_config['wandb']['name'] = "EXP00-baseline"          # ⭐️ 사용할 wandb run의 name(실험회차 이름)
loaded_config['wandb']['project'] = "nlp-dialogue-summary" # 사용할 wandb project name(프로젝트 이름)

In [14]:
# 모델이 최종 결과를 출력하기 위한 매개변수 정보를 확인합니다.
loaded_config['inference']

{'batch_size': 32,
 'ckt_path': 'model ckt path',
 'early_stopping': True,
 'generate_max_length': 50,
 'no_repeat_ngram_size': 3,
 'num_beams': 4,
 'remove_tokens': ['<usr>', '<s>', '</s>', '<pad>'],
 'result_path': './prediction/'}

### 4) 데이터 불러와서 확인해보기
- 실험에서 쓰일 데이터를 load하여 데이터의 구조와 내용을 살펴보겠습니다.
- Train, dev, test 순서대로 12457, 499, 250개 씩 데이터가 구성되어 있습니다.

In [15]:
# config에 저장된 데이터 경로를 통해 train과 validation data를 불러옵니다.
data_path = loaded_config['general']['data_path']

# train data의 구조와 내용을 확인합니다.
train_df = pd.read_csv(os.path.join(data_path,'train.csv'))
train_df.tail()

,fname,dialogue,summary,topic
12452,train_12455,#Person1#: 안녕하세요. 혹시 맨체스터에서 오신 Mr. Green 맞으신가요...,Tan Ling은 흰머리와 수염이 특징인 Mr. Green을 맞이하여 호텔로 안내합...,호텔 안내
12453,train_12456,"#Person1#: Mister Ewing이 우리 회의장에 4시에 오라고 했지, 맞...",#Person1#과 #Person2#는 Mister Ewing의 요청에 따라 회의장...,회의 준비
12454,train_12457,#Person1#: 오늘 어떻게 도와드릴까요?\n#Person2#: 차를 빌리고 싶...,#Person2#는 #Person1#의 도움으로 5일 동안 소형차를 대여합니다.,차량 대여
12455,train_12458,#Person1#: 너 오늘 좀 기분 안 좋아 보인다? 무슨 일 있어?\n#Pers...,#Person2#의 어머니가 직장을 잃으셨다. #Person2#는 어머니가 우울해하...,실직과 대처
12456,train_12459,"#Person1#: 엄마, 나 다음 주 토요일에 이모부네 가족 보러 가는데, 오늘 ...",#Person1#은 다음 주 토요일에 이모부네 가족을 방문하기 위해 짐을 싸야 하는...,가족 방문 준비


In [16]:
# validation data의 구조와 내용을 확인합니다.
val_df = pd.read_csv(os.path.join(data_path,'dev.csv'))
val_df.tail()

,fname,dialogue,summary,topic
494,dev_495,#Person1#: 새해가 되니까 나도 새 출발을 하기로 했어.\n#Person2#...,#Person1#은 새해에 담배를 끊고 커밍아웃 하기로 결심했습니다. #Person...,새해 결심
495,dev_496,#Person1#: 너 Joe랑 결혼했지?\n#Person2#: Joe? 무슨 말이...,"#Person1#은 #Person2#가 Joe와 결혼했다고 생각하지만, #Perso...",사랑과 결혼 오해
496,dev_497,"#Person1#: 어떻게 도와드릴까요, 아줌마?\n#Person2#: 제 차에서 ...","#Person2#의 차에서 소리가 나며, 브레이크 수리가 필요한 상황입니다. #Pe...",차량 소음 및 수리
497,dev_498,"#Person1#: 여보세요, 아마존 고객 서비스입니다. 어떻게 도와드릴까요?\n#...",#Person2#가 아마존 고객 서비스에 전화하여 아마존에서 구매한 책에 53페이지...,책 페이지 누락
498,dev_499,#Person1#: 벌써 여름이 다가오다니 믿기지 않아. \n#Person2#: 맞...,"#Person2#는 여름방학 동안 파티에서 일하는 회사에서 일하며, 주로 음식 준비...",여름방학 일자리


In [17]:
# test data의 구조와 내용을 확인합니다.
test_df = pd.read_csv(os.path.join(data_path,'test.csv'))
test_df.tail()

,fname,dialogue
494,test_495,"#Person1#: 얘, Charlie, 학교 끝나고 우리 집에 와서 나랑 비디오 ..."
495,test_496,#Person1#: 어떻게 시골 음악에 관심을 갖게 되었어요?\n#Person2#:...
496,test_497,"#Person1#: 저기, Alice. 여기는 처음 와봤어요. 어떻게 기계를 사용하..."
497,test_498,#Person1#: Matthew? 안녕! \n#Person2#: Steve! 진짜...
498,test_499,"#Person1#: 어, Betsy, 좋은 소식 들었어?\n#Person2#: 아니..."


In [18]:
print(f'Train: {len(train_df):,}개')
print(f'Dev  : {len(val_df):,}개')
print(f'Test : {len(test_df):,}개')
display(train_df.head(2))

Train: 12,457개
Dev  : 499개
Test : 499개


,fname,dialogue,summary,topic
0,train_0,"#Person1#: 안녕하세요, Mr. Smith. 저는 Dr. Hawkins입니다...","Mr. Smith는 Dr. Hawkins에게 건강검진을 받으러 와서, 매년 검진 필...",건강검진
1,train_1,"#Person1#: 안녕하세요, Mrs. Parker. 잘 지내셨나요?\n#Pers...","Mrs. Parker가 Ricky와 함께 백신 접종을 위해 방문하였고, Dr. Pe...",백신 접종


## 1. 데이터 가공 및 데이터셋 클래스 구축
- csv file 을 불러와서 encoder 와 decoder의 입력형태로 가공해줍니다.
- 가공된 데이터를 torch dataset class 로 구축하여 모델에 입력가능한 형태로 만듭니다.

In [19]:
# 데이터 전처리를 위한 클래스로, 데이터셋을 데이터프레임으로 변환하고 인코더와 디코더의 입력을 생성합니다.
class Preprocess:
    def __init__(self,
            bos_token: str,
            eos_token: str,
        ) -> None:

        self.bos_token = bos_token
        self.eos_token = eos_token

    @staticmethod
    # 실험에 필요한 컬럼을 가져옵니다.
    def make_set_as_df(file_path, is_train = True):
        if is_train:
            df = pd.read_csv(file_path)
            train_df = df[['fname','dialogue','summary']]
            return train_df
        else:
            df = pd.read_csv(file_path)
            test_df = df[['fname','dialogue']]
            return test_df

    # BART 모델의 입력, 출력 형태를 맞추기 위해 전처리를 진행합니다.
    def make_input(self, dataset,is_test = False):
        if is_test:
            encoder_input = dataset['dialogue']
            decoder_input = [self.bos_token] * len(dataset['dialogue'])
            return encoder_input.tolist(), list(decoder_input)
        else:
            encoder_input = dataset['dialogue']
            decoder_input = dataset['summary'].apply(lambda x : self.bos_token + str(x)) # Ground truth를 디코더의 input으로 사용하여 학습합니다.
            decoder_output = dataset['summary'].apply(lambda x : str(x) + self.eos_token)
            return encoder_input.tolist(), decoder_input.tolist(), decoder_output.tolist()


In [42]:
# Train에 사용되는 Dataset 클래스를 정의합니다.
class DatasetForTrain(Dataset):
    def __init__(self, encoder_input, decoder_input, labels, length):
        self.encoder_input = encoder_input
        self.decoder_input = decoder_input
        self.labels = labels
        self.len = length

    def __getitem__(self, idx):
        item = {k: v[idx].clone().detach() for k, v in self.encoder_input.items()} # item[input_ids], item[attention_mask]
        item2 = {k: v[idx].clone().detach() for k, v in self.decoder_input.items()} # item2[input_ids], item2[attention_mask]
        item2['decoder_input_ids'] = item2['input_ids']
        item2['decoder_attention_mask'] = item2['attention_mask']
        item2.pop('input_ids')
        item2.pop('attention_mask')
        item.update(item2) #item[input_ids], item[attention_mask] item[decoder_input_ids], item[decoder_attention_mask]
        item['labels'] = self.labels['input_ids'][idx] #item[input_ids], item[attention_mask] item[decoder_input_ids], item[decoder_attention_mask], item[labels]
        return item

    def __len__(self):
        return self.len

# Validation에 사용되는 Dataset 클래스를 정의합니다.
class DatasetForVal(Dataset):
    def __init__(self, encoder_input, decoder_input, labels, length):
        self.encoder_input = encoder_input
        self.decoder_input = decoder_input
        self.labels = labels
        self.len = length

    def __getitem__(self, idx):
        item = {k: v[idx].clone().detach() for k, v in self.encoder_input.items()} # item[input_ids], item[attention_mask]
        item2 = {k: v[idx].clone().detach() for k, v in self.decoder_input.items()} # item2[input_ids], item2[attention_mask]
        item2['decoder_input_ids'] = item2['input_ids']
        item2['decoder_attention_mask'] = item2['attention_mask']
        item2.pop('input_ids')
        item2.pop('attention_mask')
        item.update(item2) #item[input_ids], item[attention_mask] item[decoder_input_ids], item[decoder_attention_mask]
        item['labels'] = self.labels['input_ids'][idx] #item[input_ids], item[attention_mask] item[decoder_input_ids], item[decoder_attention_mask], item[labels]
        return item

    def __len__(self):
        return self.len

# Test에 사용되는 Dataset 클래스를 정의합니다.
class DatasetForInference(Dataset):
    def __init__(self, encoder_input, test_id, length):
        self.encoder_input = encoder_input
        self.test_id = test_id
        self.len = length

    def __getitem__(self, idx):
        item = {k: v[idx].clone().detach() for k, v in self.encoder_input.items()}
        item['ID'] = self.test_id[idx]
        return item

    def __len__(self):
        return self.len


In [43]:
# tokenization 과정까지 진행된 최종적으로 모델에 입력될 데이터를 출력합니다.
def prepare_train_dataset(config, preprocessor, data_path, tokenizer):
    train_file_path = os.path.join(data_path,'train.csv')
    val_file_path = os.path.join(data_path,'dev.csv')

    # train, validation에 대해 각각 데이터프레임을 구축합니다.
    train_data = preprocessor.make_set_as_df(train_file_path)
    val_data = preprocessor.make_set_as_df(val_file_path)

    print('-'*150)
    print(f'train 대화문:\n 대화: {train_data["dialogue"][0]}')
    print(f'train 요약문:\n {train_data["summary"][0]}')

    print('-'*150)
    print(f'val 대화문:\n {val_data["dialogue"][0]}')
    print(f'val 요약문:\n {val_data["summary"][0]}')

    enc_len = config['tokenizer']['encoder_max_len']
    dec_len = config['tokenizer']['decoder_max_len']

    e_train, d_train, o_train = preprocessor.make_input(train_data)
    e_val,   d_val,   o_val   = preprocessor.make_input(val_data)
    print('-'*10, 'Load data complete', '-'*10,)

    tok_e_train = tokenizer(e_train, return_tensors='pt', padding=True, truncation=True,
                            max_length=enc_len, add_special_tokens=True, return_token_type_ids=False)
    tok_d_train = tokenizer(d_train, return_tensors='pt', padding=True, truncation=True,
                            max_length=dec_len, add_special_tokens=True, return_token_type_ids=False)
    tok_o_train = tokenizer(o_train, return_tensors='pt', padding=True, truncation=True,
                            max_length=dec_len, add_special_tokens=True, return_token_type_ids=False)

    tok_e_val = tokenizer(e_val, return_tensors='pt', padding=True, truncation=True,
                          max_length=enc_len, add_special_tokens=True, return_token_type_ids=False)
    tok_d_val = tokenizer(d_val, return_tensors='pt', padding=True, truncation=True,
                          max_length=dec_len, add_special_tokens=True, return_token_type_ids=False)
    tok_o_val = tokenizer(o_val, return_tensors='pt', padding=True, truncation=True,
                          max_length=dec_len, add_special_tokens=True, return_token_type_ids=False)

    train_dataset = DatasetForTrain(tok_e_train, tok_d_train, tok_o_train, len(e_train))
    val_dataset   = DatasetForVal  (tok_e_val,   tok_d_val,   tok_o_val,   len(e_val))

    print('-'*10, 'Make dataset complete', '-'*10,)
    print(f'Train dataset: {len(train_dataset)}개 | Val dataset: {len(val_dataset)}개')
    return train_dataset, val_dataset

## 2. Trainer 및 Trainingargs 구축하기
- Huggingface 의 Trainer 와 Training arguments를 활용하여 모델 학습을 일괄적으로 처리해주는 클래스를 정의합니다.

In [44]:
def compute_metrics(config, tokenizer, pred):
    """ROUGE 평가 - MeCab 형태소 기반 (v6)"""
    import mecab
    rouge_scorer = Rouge()
    m = mecab.MeCab()

    def mecab_tokenize(text):
        tokens = [tok for tok, _ in m.pos(str(text)) if tok.strip()]
        return ' '.join(tokens) if tokens else str(text)

    predictions = pred.predictions
    labels      = pred.label_ids

    predictions[predictions == -100] = tokenizer.pad_token_id
    labels[labels == -100]           = tokenizer.pad_token_id

    decoded_preds  = tokenizer.batch_decode(
        predictions, skip_special_tokens=True, clean_up_tokenization_spaces=True)
    decoded_labels = tokenizer.batch_decode(
        labels,      skip_special_tokens=True, clean_up_tokenization_spaces=True)

    decoded_preds  = decoded_preds.copy()
    decoded_labels = decoded_labels.copy()

    remove_tokens = config['inference']['remove_tokens']
    for token in remove_tokens:
        decoded_preds  = [s.replace(token, ' ') for s in decoded_preds]
        decoded_labels = [s.replace(token, ' ') for s in decoded_labels]

    # 샘플 출력
    for i in range(min(3, len(decoded_preds))):
        print('-' * 100)
        print(f'PRED: {decoded_preds[i]}')
        print(f'GOLD: {decoded_labels[i]}')

    # ✅ MeCab 형태소 분석 후 ROUGE 계산
    preds_tok  = [mecab_tokenize(p) for p in decoded_preds]
    labels_tok = [mecab_tokenize(l) for l in decoded_labels]

    results = rouge_scorer.get_scores(preds_tok, labels_tok, avg=True)

    r1  = results['rouge-1']['f']
    r2  = results['rouge-2']['f']
    rl  = results['rouge-l']['f']
    avg = (r1 + r2 + rl) / 3
    print(f'ROUGE-1={r1:.4f} | ROUGE-2={r2:.4f} | ROUGE-L={rl:.4f} | AVG={avg:.4f}')
    return {'rouge_avg': avg}


In [45]:
# 학습을 위한 tokenizer와 사전 학습된 모델을 불러옵니다.
def load_tokenizer_and_model_for_train(config,device):
    print('-'*10, 'Load tokenizer & model', '-'*10,)
    print('-'*10, f'Model Name : {config["general"]["model_name"]}', '-'*10,)
    model_name = config['general']['model_name']
    bart_config = BartConfig.from_pretrained(model_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = BartForConditionalGeneration.from_pretrained(model_name,config=bart_config)

    special_tokens_dict={'additional_special_tokens':config['tokenizer']['special_tokens']}
    tokenizer.add_special_tokens(special_tokens_dict)

    # 사전에 special token을 추가했으므로 재구성 해줍니다.
    model.resize_token_embeddings(len(tokenizer))
    model.to(device)
    print(model.config)

    print('-'*10, 'Load tokenizer & model complete', '-'*10,)
    print(f'vocab 크기: {len(tokenizer)}')
    print(f'special_tokens 개수: {len(config["tokenizer"]["special_tokens"])}개')
    return model , tokenizer

In [46]:
# 학습을 위한 trainer 클래스와 매개변수를 정의합니다.
def load_trainer_for_train(config,generate_model,tokenizer,train_dataset,val_dataset):
    print('-'*10, 'Make training arguments', '-'*10,)
    # set training args
    training_args = Seq2SeqTrainingArguments(
                output_dir=config['general']['output_dir'], # model output directory
                overwrite_output_dir=config['training']['overwrite_output_dir'],
                num_train_epochs=config['training']['num_train_epochs'],  # total number of training epochs
                learning_rate=config['training']['learning_rate'], # learning_rate
                per_device_train_batch_size=config['training']['per_device_train_batch_size'], # batch size per device during training
                per_device_eval_batch_size=config['training']['per_device_eval_batch_size'],# batch size for evaluation
                warmup_ratio=config['training']['warmup_ratio'],  # number of warmup steps for learning rate scheduler
                weight_decay=config['training']['weight_decay'],  # strength of weight decay
                lr_scheduler_type=config['training']['lr_scheduler_type'],
                optim =config['training']['optim'],
                gradient_accumulation_steps=config['training']['gradient_accumulation_steps'],
                evaluation_strategy=config['training']['evaluation_strategy'], # evaluation strategy to adopt during training
                save_strategy =config['training']['save_strategy'],
                save_total_limit=config['training']['save_total_limit'], # number of total save model.
                fp16=config['training']['fp16'],
                load_best_model_at_end=config['training']['load_best_model_at_end'], # 최종적으로 가장 높은 점수 저장
                seed=config['training']['seed'],
                logging_dir=config['training']['logging_dir'], # directory for storing logs
                logging_strategy=config['training']['logging_strategy'],
                predict_with_generate=config['training']['predict_with_generate'], #To use BLEU or ROUGE score
                generation_max_length=config['training']['generation_max_length'],
                do_train=config['training']['do_train'],
                do_eval=config['training']['do_eval'],
                report_to=config['training']['report_to'], # (선택) wandb를 사용할 때 설정합니다.
                metric_for_best_model       = config['training']['metric_for_best_model'],
                greater_is_better           = config['training']['greater_is_better'],
            )

    # (선택) 모델의 학습 과정을 추적하는 wandb를 사용하기 위해 초기화 해줍니다.
    wandb.init(
        entity=config['wandb']['entity'],
        project=config['wandb']['project'],
        name=config['wandb']['name'],
    )

    # (선택) 모델 checkpoint를 wandb에 저장하도록 환경 변수를 설정합니다.
    os.environ["WANDB_LOG_MODEL"]="true"
    os.environ["WANDB_WATCH"]="false"

    # Validation loss가 더 이상 개선되지 않을 때 학습을 중단시키는 EarlyStopping 기능을 사용합니다.
    MyCallback = EarlyStoppingCallback(
        early_stopping_patience=config['training']['early_stopping_patience'],
        early_stopping_threshold=config['training']['early_stopping_threshold']
    )
    print('-'*10, 'Make training arguments complete', '-'*10,)
    print('-'*10, 'Make trainer', '-'*10,)

    # Trainer 클래스를 정의합니다.
    trainer = Seq2SeqTrainer(
        model=generate_model, # 사용자가 사전 학습하기 위해 사용할 모델을 입력합니다.
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics = lambda pred: compute_metrics(config,tokenizer, pred),
        callbacks = [MyCallback]
    )
    print('-'*10, 'Make trainer complete', '-'*10,)

    return trainer

## 3. 🔍 Optuna 하이퍼파라미터 자동 탐색
- **model_init** 함수로 매 trial마다 모델 새로 초기화
- **Bayesian Optimization** 방식으로 최적 조합 자동 탐색
- 탐색 완료 후 최적 하이퍼파라미터를 config에 반영하여 재학습

> ⚠️ n_trials=10 기준 약 5시간 소요 → 밤에 돌려놓기 권장  (서버실패, 3으로 진행)
> **주의** : 커널 재시작 후 이 섹션 셀들만 순서대로 실행하면 CUDA 에버 방지

실행 순서 : import -> config -> dataset -> model_init -> training_args -> trainer -> 탐색

In [25]:
# Optuna 하이퍼파라미터 자동 탐색

!pip install optuna -q
import optuna
print(f'Optuna 버전: {optuna.__version__}')


Optuna 버전: 4.7.0


In [26]:
# model_init 함수 - 매 trial마다 모델 새로 초기화

def model_init(trial=None):
    """매 trial마다 모델 새로 초기화 - Optuna 필수"""
    model_name = loaded_config["general"]["model_name"]
    bart_config = BartConfig().from_pretrained(model_name)
    model = BartForConditionalGeneration.from_pretrained(
        model_name, config=bart_config, ignore_mismatched_sizes=True)

    # # special token 추가
    # optuna_tokenizer = AutoTokenizer.from_pretrained(model_name)
    # special_tokens_dict = {"additional_special_tokens": loaded_config["tokenizer"]["special_tokens"]}
    # optuna_tokenizer.add_special_tokens(special_tokens_dict)
    # model.resize_token_embeddings(len(optuna_tokenizer))
    
    # ✅ optuna_tokenizer로 통일 (CUDA embedding 크기 불일치 방지)
    # 이미 로드된 tokenizer 재사용 (매번 새로 로드 X)
    model.resize_token_embeddings(len(optuna_tokenizer))
    return model

In [27]:
# 탐색할 하이퍼파라미터 범위 정의

def hp_space(trial):
    """탐색할 하이퍼파라미터 범위"""
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-6, 5e-5, log=True),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [16, 32]),
        "warmup_ratio": trial.suggest_float("warmup_ratio", 0.0, 0.2),
        "lr_scheduler_type": trial.suggest_categorical("lr_scheduler_type", ["linear", "cosine"]),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 5, 20),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.1),
    }


In [28]:
# Optuna용 데이터셋 준비 & dataset 준비

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
optuna_tokenizer = AutoTokenizer.from_pretrained(loaded_config["general"]["model_name"])
special_tokens_dict = {"additional_special_tokens": loaded_config["tokenizer"]["special_tokens"]}
optuna_tokenizer.add_special_tokens(special_tokens_dict)

# optuna_tokenizer로 통일
print(f"optuna_tokenizer 크기 : {len(optuna_tokenizer)}")  # 30022 확인용

preprocessor = Preprocess(
    loaded_config["tokenizer"]["bos_token"],
    loaded_config["tokenizer"]["eos_token"]
)
optuna_train_dataset, optuna_val_dataset = prepare_train_dataset(
    loaded_config, preprocessor,
    loaded_config["general"]["data_path"],
    optuna_tokenizer
)

# 토큰 ID 범위 검증
sample = optuna_train_dataset[0]
print(f'input_ids 최대값: {sample["input_ids"].max()}')  # vocab 크기 미만이어야 함
print(f'labels 최대값   : {sample["labels"][sample["labels"] != -100].max()}')

optuna_tokenizer 크기 : 30022
------------------------------------------------------------------------------------------------------------------------------------------------------
train 대화문:
 대화: #Person1#: 안녕하세요, Mr. Smith. 저는 Dr. Hawkins입니다. 오늘 무슨 일로 오셨어요? 
#Person2#: 건강검진을 받으려고 왔어요. 
#Person1#: 네, 5년 동안 검진을 안 받으셨네요. 매년 한 번씩 받으셔야 해요. 
#Person2#: 알죠. 특별히 아픈 데가 없으면 굳이 갈 필요가 없다고 생각했어요. 
#Person1#: 음, 심각한 질병을 피하려면 미리 발견하는 게 제일 좋거든요. 본인을 위해서라도 매년 한 번은 오세요. 
#Person2#: 알겠습니다. 
#Person1#: 여기 좀 볼까요. 눈과 귀는 괜찮으시네요. 깊게 숨 한 번 쉬어보세요. Mr. Smith, 담배 피우세요? 
#Person2#: 네. 
#Person1#: 담배가 폐암하고 심장병의 주된 원인인 거 아시죠? 끊으셔야 해요. 
#Person2#: 수백 번 시도했는데, 도저히 습관이 안 끊어져요. 
#Person1#: 음, 도움 될만한 수업과 약물들이 있습니다. 가시기 전에 더 정보를 드릴게요. 
#Person2#: 네, 고맙습니다, 의사 선생님.
train 요약문:
 Mr. Smith는 Dr. Hawkins에게 건강검진을 받으러 와서, 매년 검진 필요성을 안내받고 흡연 습관 개선을 위한 도움을 제안받았습니다.
------------------------------------------------------------------------------------------------------------------------------------------------------
val 대화문:
 #Person1

In [29]:
# Optuna용 TrainingArguments (고정값)

optuna_training_args = Seq2SeqTrainingArguments(
    output_dir="./outputs/optuna",
    overwrite_output_dir=True,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    predict_with_generate=True,
    generation_max_length=50,
    fp16=True,
    seed=42,
    logging_strategy="epoch",
    do_train=True,
    do_eval=True,
    report_to="none",  # Optuna 탐색 중에는 wandb 끄기
    # metric_for_best_model="eval_loss",
    metric_for_best_model="rouge_avg",
    greater_is_better=True,   # 추가
)


In [30]:
# Optuna Trainer 정의

optuna_trainer = Seq2SeqTrainer(
    args=optuna_training_args,
    train_dataset=optuna_train_dataset,
    eval_dataset=optuna_val_dataset,
    # proce=optuna_tokenizer,
    model_init=model_init,    # ✅ model_init 활용
    compute_metrics=lambda pred: compute_metrics(loaded_config, optuna_tokenizer, pred),
    callbacks=[EarlyStoppingCallback(
        early_stopping_patience=2,
        early_stopping_threshold=0.001
    )]
)

Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.


In [31]:
# Optuna 탐색 시작

print("Optuna 하이퍼파라미터 탐색 시작...")
print("⚠️ n_trials=3 기준 약 1~2시간 소요")

best_trials = optuna_trainer.hyperparameter_search(
    direction="maximize",      # ROUGE 최대화
    backend="optuna",
    hp_space=hp_space,
    n_trials=3,               # ✅ 10-> 3 탐색 횟수 (시간 여유 있으면 늘려도 됨)
)

print("=== ✅ Optuna 탐색 완료 ===")
print(f"최적 ROUGE 점수 : {best_trials.objective:.4f}")
print(f"최적 하이퍼파라미터:")
for k, v in best_trials.hyperparameters.items():
    print(f"  {k}: {v}")

[I 2026-03-10 04:36:36,523] A new study created in memory with name: no-name-6d029a46-7771-4862-b036-1ad84d98a18d
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Optuna 하이퍼파라미터 탐색 시작...
⚠️ n_trials=3 기준 약 1~2시간 소요


You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.


Epoch,Training Loss,Validation Loss,Rouge Avg
1,4.076000,1.612150,0.120647
2,1.320300,1.088908,0.163236
3,1.121800,1.054492,0.172025
4,1.063700,1.038867,0.176624
5,1.030400,1.034642,0.182691
6,1.014700,1.031302,0.181524
7,1.007700,1.031523,0.180415


----------------------------------------------------------------------------------------------------
PRED: 은 감기에 걸렸지만, 천식 검사를 위해 폐 전문의에게 가보시는 것이 좋습니다.
GOLD: 는 숨쉬기 어려워합니다. 의사는  에게 증상을 확인하고, 천식 검사를 위해 폐 전문의에게 가볼 것을 권합니다.
----------------------------------------------------------------------------------------------------
PRED: 야 Jmy는  에게  에게  에게  에게  에게  에게  에게  에게  에게  에게  
GOLD: 는 Jimmy를 운동하러 초대하고 팔과 복근 운동을 하도록 설득합니다.
----------------------------------------------------------------------------------------------------
PRED: 은 건강에 좋지 않은 음식과 과일이랑 채소, 닭고기를 먹습니다.
GOLD: 은 건강에 안 좋은 음식을 그만 먹기로 결심하고,  는 자신의 건강한 식단을  에게 공유합니다.
ROUGE-1=0.1639 | ROUGE-2=0.0419 | ROUGE-L=0.1561 | AVG=0.1206
----------------------------------------------------------------------------------------------------
PRED: 은 감기에 걸렸지만, 천식 검사를 위해 폐 전문의에게 가보라고 권유합니다.
GOLD: 는 숨쉬기 어려워합니다. 의사는  에게 증상을 확인하고, 천식 검사를 위해 폐 전문의에게 가볼 것을 권합니다.
-------------------------------------------------------------------------------------------------

...

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].
[I 2026-03-10 05:08:30,214] Trial 1 finished with value: 0.17190859422456425 and parameters: {'learning_rate': 1.412481408003982e-06, 'per_device_train_batch_size': 16, 'warmup_ratio': 0.017649500023780164, 'lr_scheduler_type': 'cosine', 'num_train_epochs': 18, 'weight_decay': 0.05094144116526718}. Best is trial 0 with value: 0.180414847706221.
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.


Epoch,Training Loss,Validation Loss,Rouge Avg
1,2.761300,1.101879,0.162318
2,1.114800,1.022526,0.180661
3,0.969200,0.995829,0.189501
4,0.837100,0.996080,0.185194
5,0.730000,1.013215,0.195945
6,0.642600,1.034373,0.190718
7,0.573000,1.058547,0.194290


----------------------------------------------------------------------------------------------------
PRED: 은  에게 감기에 걸렸다고 말합니다.  는 천식 검사를 위해 폐 전문의에게 가보라고 조언합니다.
GOLD: 는 숨쉬기 어려워합니다. 의사는  에게 증상을 확인하고, 천식 검사를 위해 폐 전문의에게 가볼 것을 권합니다.
----------------------------------------------------------------------------------------------------
PRED: 야 Jimmy는 농구 때문에 다리가 아파서  에게 운동 날짜를 바꾸자고 제안한다.  은  에게  의 주간 일정을 설명한다.
GOLD: 는 Jimmy를 운동하러 초대하고 팔과 복근 운동을 하도록 설득합니다.
----------------------------------------------------------------------------------------------------
PRED: 은 건강에 좋지 않은 음식을 줄이고 싶어 합니다.  는 과일과 채소를 많이 먹고, 닭고기는 구워 먹습니다.
GOLD: 은 건강에 안 좋은 음식을 그만 먹기로 결심하고,  는 자신의 건강한 식단을  에게 공유합니다.
ROUGE-1=0.2209 | ROUGE-2=0.0562 | ROUGE-L=0.2098 | AVG=0.1623
----------------------------------------------------------------------------------------------------
PRED: 은  에게 감기에 걸렸다고 말합니다.  는 천식 검사를 위해 폐 전문의에게 가보라고 조언합니다.
GOLD: 는 숨쉬기 어려워합니다. 의사는  에게 증상을 확인하고, 천식 검사를 위해 폐 전문의에게 가볼 것을 권합니다.
-------------------------------

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].
[I 2026-03-10 05:23:03,526] Trial 2 finished with value: 0.19428967639611938 and parameters: {'learning_rate': 2.1565986120357166e-05, 'per_device_train_batch_size': 16, 'warmup_ratio': 0.18592277168762014, 'lr_scheduler_type': 'cosine', 'num_train_epochs': 11, 'weight_decay': 0.023269388734818465}. Best is trial 2 with value: 0.19428967639611938.


=== ✅ Optuna 탐색 완료 ===
최적 ROUGE 점수 : 0.1943
최적 하이퍼파라미터:
  learning_rate: 2.1565986120357166e-05
  per_device_train_batch_size: 16
  warmup_ratio: 0.18592277168762014
  lr_scheduler_type: cosine
  num_train_epochs: 11
  weight_decay: 0.023269388734818465


In [32]:
# ✅ best 파라미터 자동으로 config에 반영
print('=== 📌 best 파라미터 -> config 자동 반영 ===')
for k, v in best_trials.hyperparameters.items():
    if k in loaded_config['training']:
        old = loaded_config['training'][k]
        loaded_config['training'][k] = v
        print(f'  {k}: {old} -> {v}')

# config yaml 재저장
with open(config_path, 'w') as f:
    yaml.dump(loaded_config, f, allow_unicode=True, default_flow_style=False)
print('\nconfig 재저장 완료! main() 실행하면 best 파라미터로 풀 학습 시작')

=== 📌 best 파라미터 -> config 자동 반영 ===
  learning_rate: 1e-05 -> 2.1565986120357166e-05
  per_device_train_batch_size: 32 -> 16
  warmup_ratio: 0.1 -> 0.18592277168762014
  lr_scheduler_type: cosine -> cosine
  num_train_epochs: 20 -> 11
  weight_decay: 0.01 -> 0.023269388734818465

config 재저장 완료! main() 실행하면 best 파라미터로 풀 학습 시작


In [33]:
# # 최적값 config에 반영 방법 안내
# print("=== 📌 다음 실험에 반영하는 방법 ===")
# print('config_data["training"]["learning_rate"] =', best_trials.hyperparameters["learning_rate"])
# print('config_data["training"]["warmup_ratio"] =', best_trials.hyperparameters["warmup_ratio"])
# print('config_data["training"]["lr_scheduler_type"] =', best_trials.hyperparameters["lr_scheduler_type"])
# print('config_data["training"]["per_device_train_batch_size"] =', best_trials.hyperparameters["per_device_train_batch_size"])


In [34]:
# print(f"최적 ROUGE: {best_trials.objective:.4f}")
# print(f"최적 파라미터: ")
# for k, v in best_trials.hyperparameters.items():
#     print(f"{k}:{v}")

## 4. 모델 학습하기 (main)

> Optuna 완료 후 best 파라미터가 config에 자동 반영된 상태에서 실행
- 앞에서 구축한 클래스 및 함수를 활용하여 학습 진행합니다.

In [35]:
def main(config):
    # 사용할 device를 정의합니다.
    device = torch.device('cuda:0' if torch.cuda.is_available()  else 'cpu')
    print('-'*10, f'device : {device}', '-'*10,)
    print(torch.__version__)

    # 사용할 모델과 tokenizer를 불러옵니다.
    model , tokenizer = load_tokenizer_and_model_for_train(config,device)
    print('-'*10,"tokenizer special tokens : ",tokenizer.special_tokens_map,'-'*10)

    # 학습에 사용할 데이터셋을 불러옵니다.
    preprocessor = Preprocess(config['tokenizer']['bos_token'], config['tokenizer']['eos_token']) # decoder_start_token: str, eos_token: str
    data_path = config['general']['data_path']
    train_dataset, val_dataset = prepare_train_dataset(config,preprocessor, data_path, tokenizer)

    # Trainer 클래스를 불러옵니다.
    trainer = load_trainer_for_train(config, model,tokenizer,train_dataset,val_dataset)
    trainer.train()   # 모델 학습을 시작합니다.

    # (선택) 모델 학습이 완료된 후 wandb를 종료합니다.
    wandb.finish()
    print('학습 완료!')

In [36]:
if __name__ == "__main__":
    main(loaded_config)

---------- device : cuda:0 ----------
2.1.0
---------- Load tokenizer & model ----------
---------- Model Name : digit82/kobart-summarization ----------


You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.


BartConfig {
  "_name_or_path": "digit82/kobart-summarization",
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_bias_logits": false,
  "add_final_layer_norm": false,
  "architectures": [
    "BartForConditionalGeneration"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classif_dropout": 0.1,
  "classifier_dropout": 0.1,
  "d_model": 768,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 3072,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 6,
  "decoder_start_token_id": 2,
  "do_blenderbot_90_layernorm": false,
  "dropout": 0.1,
  "encoder_attention_heads": 16,
  "encoder_ffn_dim": 3072,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 6,
  "eos_token_id": 1,
  "extra_pos_embeddings": 2,
  "force_bos_token_to_be_generated": false,
  "forced_eos_token_id": 2,
  "id2label": {
    "0": "NEGATIVE",
    "1": "POSITIVE"
  },
  "init_std": 0.02,
  "is_encoder_decoder": true,
  "label2id": {
    "NEGATIVE": 0,
    "POSITIVE": 1
  },
  "max_position_embeddings":

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


---------- Make dataset complete ----------
Train dataset: 12457개 | Val dataset: 499개
---------- Make training arguments ----------


...

Epoch,Training Loss,Validation Loss,Rouge Avg
1,2.789500,1.102313,0.161102
2,1.116200,1.024540,0.181122
3,0.971100,0.996214,0.186658
4,0.838600,0.997099,0.189192
5,0.731400,1.015478,0.193497
6,0.644300,1.037659,0.194122
7,0.574400,1.061143,0.200189
8,0.519400,1.077231,0.199283
9,0.485400,1.092005,0.192944
10,0.463400,1.097647,0.191715


----------------------------------------------------------------------------------------------------
PRED: 은  에게 감기에 걸렸다고 말합니다.  는 천식 검사를 위해 폐 전문의에게 가보라고 권유합니다.
GOLD: 는 숨쉬기 어려워합니다. 의사는  에게 증상을 확인하고, 천식 검사를 위해 폐 전문의에게 가볼 것을 권합니다.
----------------------------------------------------------------------------------------------------
PRED: 은 야 Jimmy에게 운동하러 가자고 제안하지만, Jimmy는  에게 운동 날짜를 바꾸자고 제안한다.
GOLD: 는 Jimmy를 운동하러 초대하고 팔과 복근 운동을 하도록 설득합니다.
----------------------------------------------------------------------------------------------------
PRED: 는 건강에 좋지 않은 음식을 줄이고 싶어 한다.  은 과일과 채소, 닭고기를 먹으며,  는 닭고기를 구워 먹기로 한다.
GOLD: 은 건강에 안 좋은 음식을 그만 먹기로 결심하고,  는 자신의 건강한 식단을  에게 공유합니다.
ROUGE-1=0.2192 | ROUGE-2=0.0562 | ROUGE-L=0.2078 | AVG=0.1611
----------------------------------------------------------------------------------------------------
PRED: 는  에게 감기에 걸렸다고 말합니다.  는 천식 검사를 위해 폐 전문의에게 가보라고 조언합니다.
GOLD: 는 숨쉬기 어려워합니다. 의사는  에게 증상을 확인하고, 천식 검사를 위해 폐 전문의에게 가볼 것을 권합니다.
------------------------------------

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


eval/loss,█▃▁▁▂▄▅▆▇█
eval/rouge_avg,▁▅▆▆▇▇██▇▆
eval/runtime,▁▆▆▅▅▆▇▅█▆
eval/samples_per_second,█▃▃▄▄▃▂▄▁▃
eval/steps_per_second,█▃▃▄▄▃▂▄▁▃
train/epoch,▁▁▂▂▃▃▃▃▄▄▅▅▆▆▆▆▇▇███
train/global_step,▁▁▂▂▃▃▃▃▄▄▅▅▆▆▆▆▇▇███
train/learning_rate,▄██▇▆▅▄▃▂▁
train/loss,█▃▃▂▂▂▁▁▁▁
train/total_flos,▁
+4,...


학습 완료!


## 5. Test 추론 & 제출파일 생성

In [37]:
# 이곳에 내가 사용할 wandb config 설정
#loaded_config['inference']['ckt_path'] = "./checkpoint-2500" #추론에 사용할 ckt 경로 설정

# ⭐️ 추론 전 checkpoint 경로 설정
# 생성된 checkpoint 폴더 중 가장 최신 것을 자동으로 찾아서 할당
checkpoints = sorted(
    glob("./checkpoint-*"),
    key=lambda x: int(x.split("-")[1]) # 숫자 기준 정렬
)

if checkpoints:
    latest_ckt = checkpoints[-1] # 가장 마지막 = 가장 최신
    loaded_config['inference']['ckt_path'] = latest_ckt
    print(f"✅ 자동 할당된 checkpoint: {latest_ckt}")
else :
    print("❌ checkpoint 폴더를 찾을 수 없습니다. 학습이 완료되었는지 확인하세요.")

# Early Stopping으로 학습이 중간에 멈춰도 자동으로 마지막 checkpoint 잡아줘요.


✅ 자동 할당된 checkpoint: ./checkpoint-7790


- test data를 사용하여 모델의 성능을 확인합니다.

In [38]:
# tokenization 과정까지 진행된 최종적으로 모델에 입력될 데이터를 출력합니다.
def prepare_test_dataset(config,preprocessor, tokenizer):

    test_file_path = os.path.join(config['general']['data_path'],'test.csv')

    test_data = preprocessor.make_set_as_df(test_file_path,is_train=False)

    print('-'*150)
    print(f'test_data:\n{test_data["dialogue"][0]}')
    print('-'*150)

    e_test, d_test = preprocessor.make_input(test_data, is_test=True)
    print('-'*10, 'Load data complete', '-'*10,)

    tok_e_test = tokenizer(
        e_test, return_tensors='pt', padding=True, truncation=True,
        max_length=config['tokenizer']['encoder_max_len'],
        add_special_tokens=True, return_token_type_ids=False
    )

    test_dataset = DatasetForInference(tok_e_test, test_data['fname'], len(e_test))
    print('-'*10, 'Make dataset complete', '-'*10,)

    return test_data, test_dataset

In [39]:
# 추론을 위한 tokenizer와 학습시킨 모델을 불러옵니다.
def load_tokenizer_and_model_for_test(config,device):
    print('-'*10, 'Load tokenizer & model', '-'*10,)

    model_name = config['general']['model_name']
    ckt_path = config['inference']['ckt_path']
    print('-'*10, f'Model Name : {model_name}', '-'*10,)
    print(f'checkpoint: {ckt_path}')

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    special_tokens_dict = {'additional_special_tokens': config['tokenizer']['special_tokens']}
    tokenizer.add_special_tokens(special_tokens_dict)

    model = BartForConditionalGeneration.from_pretrained(ckt_path)
    model.resize_token_embeddings(len(tokenizer))
    model.to(device)
    print('-'*10, 'Load tokenizer & model complete', '-'*10,)

    return model , tokenizer

In [47]:
# 학습된 모델이 생성한 요약문의 출력 결과를 보여줍니다.
def inference(config):
    """test.csv 추론 -> EXP번호_ckt번호_날짜.csv 저장"""
    device = torch.device('cuda:0' if torch.cuda.is_available()  else 'cpu')
    print('-'*10, f'device : {device}', '-'*10,)
    print(torch.__version__)

    model , tokenizer = load_tokenizer_and_model_for_test(config,device)

    preprocessor = Preprocess(config['tokenizer']['bos_token'], config['tokenizer']['eos_token'])

    test_data, test_dataset = prepare_test_dataset(config,preprocessor, tokenizer)
    dataloader = DataLoader(test_dataset, batch_size=config['inference']['batch_size'])

    summaries = []
    text_ids = []
    with torch.no_grad():
        for item in tqdm(dataloader):
            text_ids.extend(item['ID'])
            generated_ids = model.generate(input_ids=item['input_ids'].to(device),
                            no_repeat_ngram_size=config['inference']['no_repeat_ngram_size'],
                            early_stopping=config['inference']['early_stopping'],
                            max_length=config['inference']['generate_max_length'],
                            num_beams=config['inference']['num_beams'],
                        )
            for ids in generated_ids:
                result = tokenizer.decode(ids)
                summaries.append(result)

    # 정확한 평가를 위하여 노이즈에 해당되는 스페셜 토큰을 제거합니다.
    # ✅ 수정
    remove_tokens = config['inference']['remove_tokens']
    result = summaries.copy()   # ✅ 초기화 추가
    for token in remove_tokens:
        result = [s.replace(token, ' ') for s in result]

    # ✅ 특수토큰 주변 불필요한 공백 제거
    result = [re.sub(r'(#\w+#)\s+', r'\1', s) for s in result]
    result = [re.sub(r'\s+', ' ', s).strip() for s in result]

    output = pd.DataFrame(
        {
            "fname": test_data['fname'],
            "summary" : result,
        }
    )

    ## ✅ 파일명(실험이름) 자동 생성 (EXP번호_ckt번호_날짜시간.csv)
    # checkpoint 번호 추출
    ckt_number = os.path.basename(config['inference']['ckt_path']).split('-')[-1]  # "2250"
    exp_number = config.get('exp_number', '00') # 실험 번호 (config에서만 관리) "00", "01", ...
    timestamp = datetime.now().strftime("%m%d_%H%M") # 타임스탬프
    filename = f"EXP{exp_number}_ckt{ckt_number}_{timestamp}.csv" # 최종파일명

    result_path = config['inference']['result_path']
    os.makedirs(result_path, exist_ok=True)
    save_path = os.path.join(result_path, filename)
    output.to_csv(save_path, index=False)
    print(f"✅ 저장완료 -> {save_path}")
    print(f'총 {len(output)}개 | 고유 요약: {output["summary"].nunique()}개')
    
    return output

In [48]:
# 학습된 모델의 test를 진행합니다.
if __name__ == "__main__":
    output = inference(loaded_config)

---------- device : cuda:0 ----------
2.1.0
---------- Load tokenizer & model ----------
---------- Model Name : digit82/kobart-summarization ----------
checkpoint: ./checkpoint-7790


You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.


---------- Load tokenizer & model complete ----------
------------------------------------------------------------------------------------------------------------------------------------------------------
test_data:
#Person1#: Ms. Dawson, 받아쓰기 좀 부탁드려야겠어요. 
#Person2#: 네, 말씀하세요... 
#Person1#: 이걸 오늘 오후까지 모든 직원들에게 사내 메모로 보내야 해요. 준비됐나요? 
#Person2#: 네, 말씀하세요. 
#Person1#: 모든 직원에게 알립니다... 즉시 발효되어 모든 사내 통신은 이메일과 공식 메모로만 제한됩니다. 근무 시간 동안 즉시 메시지 프로그램 사용은 금지됩니다. 
#Person2#: 이 정책이 사내 통신에만 적용되나요, 아니면 외부 통신에도 해당되나요? 
#Person1#: 이는 모든 통신에 적용됩니다. 사무실 내 직원 간 통신 뿐만 아니라 외부 통신도 해당됩니다. 
#Person2#: 하지만 많은 직원들이 고객과 소통하려고 즉시 메시지를 사용합니다. 
#Person1#: 통신 방법을 바꿔야 할 것입니다. 이 사무실에서는 즉시 메시지를 사용하는 것을 원하지 않습니다. 너무 많은 시간이 낭비됩니다! 이제 계속해서 메모를 작성해 주세요. 어디까지 했죠? 
#Person2#: 내외부 통신에 적용됩니다. 
#Person1#: 네. 즉시 메시지를 계속 사용하면 경고 후 시정 조치가 이루어지며, 두 번째 위반 시 해고될 수 있습니다. 이번 정책에 관한 질문은 부서장에게 문의하세요. 
#Person2#: 그게 다인가요? 
#Person1#: 네. 오늘 오후 4시까지 이 메모를 작성하고 배포해주세요.
----------------------------------------------------------------------------

100%|██████████| 16/16 [00:21<00:00,  1.34s/it]

✅ 저장완료 -> ./prediction/EXP00_ckt7790_0310_0613.csv
총 499개 | 고유 요약: 499개


In [49]:
output  # 각 대화문에 대한 요약문이 출력됨을 확인할 수 있습니다.

,fname,summary
0,test_0,Ms. Dawson은 #Person1#에게 사내 메모를 작성하고 배포해 달라고 요청...
1,test_1,#Person1#과 #Person2#는 교통체증으로 인해 출퇴근에 어려움을 겪고 있...
2,test_2,#Person1#은 Kate에게 Masha와 Hero가 별거 중이어서 이혼을 당했다...
3,test_3,"#Person1#은 Brian에게 생일 선물로 목걸이를 주며, 파티에서 그녀의 인기..."
4,test_4,#Person1#과 #Person2#는 올림픽 공원의 크기와 시설에 대해 이야기하고...
...,...,...
494,test_495,Jack은 Charlie에게 공항에서 픽업해야 하지만 2시간 동안 있을 수 있으니 ...
495,test_496,#Person2#는 시골 음악에 관심을 가지게 된 계기와 라디오 방송국에서 일하게 ...
496,test_497,Alice는 #Person1#에게 세탁기에 비누가 들어 있지 않다고 설명합니다. #...
497,test_498,"Matthew와 Steve는 임대를 갱신하고 싶지 않아 집을 찾고 있으며, 그녀의 ..."


## 6. dev.csv 추론 & 로컬 ROUGE 평가용
> 리더보드 제출 전 로컬 점수 확인용
- dev.csv의 정답 요약과 모델 예측 요약을 비교해서 ROUGE 점수 계산

In [50]:
def inference_dev(config):
    """dev.csv 추론 → 로컬 ROUGE 평가용 (리더보드 제출 전 점수 확인)"""
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    generate_model, tokenizer = load_tokenizer_and_model_for_test(config, device)

    preprocessor  = Preprocess(config['tokenizer']['bos_token'], config['tokenizer']['eos_token'])
    dev_file_path = os.path.join(config['general']['data_path'], 'dev.csv')
    dev_data      = preprocessor.make_set_as_df(dev_file_path, is_train=False)

    e_dev, d_dev = preprocessor.make_input(dev_data, is_test=True)

    tok_e_dev = tokenizer(
        e_dev, return_tensors="pt", padding=True,
        add_special_tokens=True, truncation=True,
        max_length=config['tokenizer']['encoder_max_len'],
        return_token_type_ids=False
    )
    dev_dataset = DatasetForInference(tok_e_dev, dev_data['fname'], len(e_dev))
    dev_loader  = DataLoader(dev_dataset, batch_size=config['inference']['batch_size'])

    summaries, text_ids = [], []
    with torch.no_grad():
        for item in tqdm(dev_loader):
            text_ids.extend(item['ID'])
            generated_ids = generate_model.generate(
                input_ids=item['input_ids'].to('cuda:0'),
                no_repeat_ngram_size=config['inference']['no_repeat_ngram_size'],
                early_stopping=config['inference']['early_stopping'],
                max_length=config['inference']['generate_max_length'],
                num_beams=config['inference']['num_beams'],
            )
            for ids in generated_ids:
                summaries.append(tokenizer.decode(ids))

    remove_tokens = config['inference']['remove_tokens']
    result = summaries.copy()

    for token in remove_tokens:
        result = [s.replace(token, " ") for s in result]

    # ✅ 특수토큰 주변 불필요한 공백 제거
    result = [re.sub(r'(#\w+#)\s+', r'\1', s) for s in result]
    result = [re.sub(r'\s+', ' ', s).strip() for s in result]

    output = pd.DataFrame({"fname": dev_data['fname'], "summary": result})

    # ✅ 파일명에 dev_ 접두사 추가
    ckt_number = os.path.basename(config['inference']['ckt_path']).split('-')[-1]
    exp_number = config.get('exp_number', '00')
    timestamp  = datetime.now().strftime("%m%d_%H%M")
    filename   = f"dev_EXP{exp_number}_ckt{ckt_number}_{timestamp}.csv"

    result_path = config['inference']['result_path']
    os.makedirs(result_path, exist_ok=True)
    output.to_csv(os.path.join(result_path, filename), index=False)
    print(f"✅ dev 추론 완료 → {result_path}{filename}")
    return output

In [51]:
if __name__ == "__main__":
    dev_output = inference_dev(loaded_config)

---------- Load tokenizer & model ----------
---------- Model Name : digit82/kobart-summarization ----------
checkpoint: ./checkpoint-7790


You passed along `num_labels=3` with an incompatible id to label map: {'0': 'NEGATIVE', '1': 'POSITIVE'}. The number of labels wil be overwritten to 2.


---------- Load tokenizer & model complete ----------


100%|██████████| 16/16 [00:21<00:00,  1.32s/it]

✅ dev 추론 완료 → ./prediction/dev_EXP00_ckt7790_0310_0629.csv


## 6. 로컬 ROUGE 평가(LB 제출 전 점수확인)

In [52]:
def evaluate_on_dev(config, output_csv_path):
    """dev.csv 정답과 모델 예측을 비교해서 ROUGE 점수 계산 - MeCab 기반 (v6)"""
    import mecab
    rouge_scorer = Rouge()
    m = mecab.MeCab()

    def mecab_tokenize(text):
        tokens = [tok for tok, _ in m.pos(str(text)) if tok.strip()]
        return ' '.join(tokens) if tokens else str(text)

    dev_df  = pd.read_csv(os.path.join(config['general']['data_path'], 'dev.csv'))
    pred_df = pd.read_csv(output_csv_path)

    dev_df  = dev_df.sort_values('fname').reset_index(drop=True)
    pred_df = pred_df.sort_values('fname').reset_index(drop=True)

    sum_cols = [c for c in dev_df.columns if 'summary' in c.lower()]
    print(f"참조 요약 컬럼: {sum_cols}")

    predictions = pred_df['summary'].tolist()
    all_scores  = []

    for col in sum_cols:
        references  = dev_df[col].tolist()
        valid_pairs = [
            (mecab_tokenize(p), mecab_tokenize(r))
            for p, r in zip(predictions, references)
            if str(p).strip() and str(r).strip()
        ]
        preds_tok = [p for p, r in valid_pairs]
        refs_tok  = [r for p, r in valid_pairs]

        scores = rouge_scorer.get_scores(preds_tok, refs_tok, avg=True)
        all_scores.append(scores)
        print(f"\n[{col}]")
        print(f"  ROUGE-1 F1: {scores['rouge-1']['f']:.4f}")
        print(f"  ROUGE-2 F1: {scores['rouge-2']['f']:.4f}")
        print(f"  ROUGE-L F1: {scores['rouge-l']['f']:.4f}")

    avg_r1 = sum(s['rouge-1']['f'] for s in all_scores) / len(all_scores)
    avg_r2 = sum(s['rouge-2']['f'] for s in all_scores) / len(all_scores)
    avg_rl = sum(s['rouge-l']['f'] for s in all_scores) / len(all_scores)
    final_score = (avg_r1 + avg_r2 + avg_rl) / 3

    print(f"\n{'='*40}")
    print(f"📊 최종 평균 ROUGE 점수 (대회 방식 - MeCab 기반)")
    print(f"{'='*40}")
    print(f"  ROUGE-1 : {avg_r1:.4f}")
    print(f"  ROUGE-2 : {avg_r2:.4f}")
    print(f"  ROUGE-L : {avg_rl:.4f}")
    print(f"  평균    : {final_score:.4f}  ← 리더보드 점수와 유사")
    print(f"{'='*40}")
    print(f"\n📌 참고: dev는 참조 요약 1개 기준이라 리더보드(3개 기준)와 절대값 차이가 있을 수 있어요.")
    print(f"         상대적인 점수 변화 추이로 실험 효과를 판단하는 용도로 활용하세요!")
    return final_score


In [53]:
# ✅ 가장 최근 dev 추론 결과 자동으로 찾아서 평가
result_path = loaded_config['inference']['result_path']
dev_files   = sorted([f for f in os.listdir(result_path) if f.startswith('dev_')])

if dev_files:
    latest_dev_file = os.path.join(result_path, dev_files[-1])
    print(f"평가 파일: {latest_dev_file}")
    evaluate_on_dev(loaded_config, latest_dev_file)
else:
    print("❌ dev 추론 결과 파일이 없습니다. inference_dev를 먼저 실행해주세요.")

평가 파일: ./prediction/dev_EXP00_ckt7790_0310_0629.csv
참조 요약 컬럼: ['summary']

[summary]
  ROUGE-1 F1: 0.3389
  ROUGE-2 F1: 0.1438
  ROUGE-L F1: 0.3110

📊 최종 평균 ROUGE 점수 (대회 방식)
  ROUGE-1 : 0.3389
  ROUGE-2 : 0.1438
  ROUGE-L : 0.3110
  평균    : 0.2646  ← 리더보드 점수와 유사

📌 참고: dev는 참조 요약 1개 기준이라 리더보드(3개 기준)와 절대값 차이가 있을 수 있어요.
         상대적인 점수 변화 추이로 실험 효과를 판단하는 용도로 활용하세요!


## 7. 실패 케이스 분석
> 어디서 틀리는지 파악 -> Solar API 개선 방향 결정에도 활용

In [54]:
def analyze_failures(config, dev_pred_path, top_n=10):
    """ROUGE 낮은 케이스 분석"""
    rouge   = Rouge()
    dev_df  = pd.read_csv(os.path.join(config['general']['data_path'], 'dev.csv'))
    pred_df = pd.read_csv(dev_pred_path)

    merged = pd.merge(dev_df, pred_df, on='fname', suffixes=('_gold', '_pred'))

    def safe_rouge1(pred, gold):
        try:
            return rouge.get_scores(str(pred), str(gold))[0]['rouge-1']['f']
        except:
            return 0.0

    gold_col = [c for c in merged.columns if 'summary' in c and 'gold' in c]
    pred_col = [c for c in merged.columns if 'summary' in c and 'pred' in c]
    if not gold_col: gold_col = ['summary_x']
    if not pred_col: pred_col = ['summary_y']

    merged['rouge1'] = merged.apply(
        lambda r: safe_rouge1(r[pred_col[0]], r[gold_col[0]]), axis=1
    )
    merged['pred_len'] = merged[pred_col[0]].astype(str).apply(lambda x: len(x.split()))
    merged['gold_len'] = merged[gold_col[0]].astype(str).apply(lambda x: len(x.split()))

    print('='*60)
    print('실패 케이스 분석')
    print('='*60)
    print(f'전체 평균 ROUGE-1: {merged["rouge1"].mean():.4f}')
    print(f'예측 평균 길이   : {merged["pred_len"].mean():.1f}단어')
    print(f'정답 평균 길이   : {merged["gold_len"].mean():.1f}단어')

    # 반복률 분석
    def repetition_ratio(text):
        words = str(text).split()
        return 1 - len(set(words)) / len(words) if words else 0
    merged['repetition'] = merged[pred_col[0]].apply(repetition_ratio)
    print(f'예측 반복률 평균 : {merged["repetition"].mean():.3f}')
    if merged['repetition'].mean() > 0.1:
        print('-> 반복 10% 이상! no_repeat_ngram_size 증가 고려')

    # 하위 케이스 출력
    sorted_df = merged.sort_values('rouge1')
    print(f'\n하위 {top_n}개 실패 케이스')
    print('-'*60)
    for _, row in sorted_df.head(top_n).iterrows():
        print(f'\n[{row["fname"]}] ROUGE-1={row["rouge1"]:.3f}')
        print(f'정답: {row[gold_col[0]]}')
        print(f'예측: {row[pred_col[0]]}')
        print('-'*60)

    return merged

In [55]:
# 가장 최근 dev 파일로 분석
if dev_files:
    latest = os.path.join(result_path, dev_files[-1])
    failure_df = analyze_failures(loaded_config, latest, top_n=10)
else:
    print('dev 추론 파일 없음! inference_dev() 먼저 실행하세요.')

실패 케이스 분석
전체 평균 ROUGE-1: 0.2538
예측 평균 길이   : 15.3단어
정답 평균 길이   : 15.3단어
예측 반복률 평균 : 0.022

하위 10개 실패 케이스
------------------------------------------------------------

[dev_355] ROUGE-1=0.000
정답: #Person1#과 #Person2#는 스쿠터의 전 세계적 인기, 사용 방법 및 장점에 대해 이야기하고 있습니다.
예측: #Person1#은 #Person2#에게 독일 엔지니어들이 만든 스쿠터를 소개합니다. 이 스쿠터는 디자인이 예쁘고 정교하며 휴대하기 편리합니다.
------------------------------------------------------------

[dev_109] ROUGE-1=0.000
정답: #Person1#과 #Person2#는 로마인의 생활 방식과 음식 문화에 대해 이야기합니다.
예측: #Person1#은 #Person2#에게 로마인들이 차나 버스로 출근하며, 삶을 즐기고, 맛있는 음식을 좋아한다고 설명합니다.
------------------------------------------------------------

[dev_465] ROUGE-1=0.000
정답: #Person1#과 #Person2# 모두 케이트의 새 남자친구 이야기로 인해 방해받았다.
예측: #Person1#은 #Person2#에게 케이트가 어젯밤에 전화해 흥분해서 잠을 못 자겠다고 하더라고 전합니다. #Person2#는 #Person1#에게 그 전화가 저번 주에 있었던 일이라고 말합니다.
------------------------------------------------------------

[dev_461] ROUGE-1=0.000
정답: #Person1#는 #Person2#의 사촌이 농담을 즐기지만 따뜻한 마음을 가진 사람이라고 생각합니다.
예측: #Person1#은 #Person2#에게 로사리오에